# LSTM point-scale RZSM prediction

This notebook loads the two ensemble bundles produced by `LSTM_train.ipynb` and predicts RZSM for the Train, Test, and Excluded point cohorts. It requires bundles trained with the matched FNO/LSTM protocol and rejects older LSTM bundles.

Each point output contains only the source SSM, observed in-situ RZSM, ten-seed ensemble mean, and ensemble sample standard deviation. Individual seed predictions, embedded exponential-filter output, plotting files, and separate run manifests are not saved.

In [ ]:
import os

from config import configure_runtime

configure_runtime()

import pandas as pd
from IPython.display import display

## 1. Paths, settings, and reusable imports

In [ ]:
from config import (
    LSTM_PREDICTION_WORKERS,
    base_FP,
    cpuserver_data,
    das_FP,
    george_FP,
    nas_FP,
)
from LSTM.prediction import run_prediction_tasks
from LSTM.settings import (
    AREAS,
    PRODUCTS,
    PRODUCT_SETTINGS,
    STATIC_FEATURES,
    validate_static_columns,
)
from LSTM.training import TRAINING_RECIPE_VERSION, load_ensemble_bundle

from config import figures_FP, results_FP


In [ ]:
DATA_PRODUCTS = PRODUCTS
PREDICTION_AREAS = AREAS
PREDICTION_WORKERS = LSTM_PREDICTION_WORKERS

ISMN_RESULT_FP = os.path.join(results_FP, "ISMN"
)
LSTM_RESULT_FP = os.path.join(results_FP, "LSTM"
)
TRAIN_RESULT_FP = os.path.join(LSTM_RESULT_FP, "Train")
INPUT_CSV = {
    (area, product): os.path.join(
        LSTM_RESULT_FP, area, f"LSTM_input_{product}.csv"
    )
    for area in PREDICTION_AREAS
    for product in DATA_PRODUCTS
}
BUNDLE_FILE = {
    product: os.path.join(TRAIN_RESULT_FP, f"LSTM_{product}_ensemble.pt")
    for product in DATA_PRODUCTS
}

for product, bundle_file in BUNDLE_FILE.items():
    if not os.path.exists(bundle_file):
        raise FileNotFoundError(
            f"Missing {product} bundle: {bundle_file}. Run LSTM_train.ipynb first."
        )
    load_ensemble_bundle(bundle_file)

print(f"Required LSTM training recipe: {TRAINING_RECIPE_VERSION}")
print(f"Prediction workers: {PREDICTION_WORKERS}")
print(f"Products: {DATA_PRODUCTS}; areas: {PREDICTION_AREAS}")

## 2. Predict each point cohort

Predictions are emitted only at finite source-SSM timestamps after 32 valid observations have accumulated. Earlier dates and non-observation dates remain `NaN`. Existing files for the same pixels are replaced atomically; unrelated files are not removed.

In [ ]:
prediction_summaries = []

for area in PREDICTION_AREAS:
    for product in DATA_PRODUCTS:
        input_file = INPUT_CSV[(area, product)]
        if not os.path.exists(input_file):
            if area == "Excluded":
                continue
            raise FileNotFoundError(f"Missing point inventory: {input_file}")

        input_frame = pd.read_csv(input_file)
        if input_frame.empty:
            continue
        validate_static_columns(input_frame.columns, input_file)
        output_directory = os.path.join(
            LSTM_RESULT_FP, area, "RZSM_prediction", product
        )
        os.makedirs(output_directory, exist_ok=True)
        point_directory = os.path.join(
            ISMN_RESULT_FP, f"ISMN_{area}", PRODUCT_SETTINGS[product]["folder"]
        )

        tasks = []
        for row in input_frame.to_dict(orient="records"):
            pixel = f"{int(row['lat_idx'])}_{int(row['lon_idx'])}"
            tasks.append(
                {
                    "row": row,
                    "point_file": os.path.join(point_directory, f"{pixel}.nc"),
                    "output_file": os.path.join(
                        output_directory, f"{pixel}_prediction.nc"
                    ),
                }
            )

        results = run_prediction_tasks(
            tasks=tasks,
            bundle_file=BUNDLE_FILE[product],
            product=product,
            workers=PREDICTION_WORKERS,
        )
        failures = [result for result in results if result["status"] != "succeeded"]
        prediction_summaries.append(
            {
                "area": area,
                "product": product,
                "input_pixels": len(tasks),
                "succeeded": len(results) - len(failures),
                "failed": len(failures),
                "output_directory": output_directory,
            }
        )
        if failures:
            preview = "; ".join(
                f"{item['pixel']}: {item['reason']}" for item in failures[:5]
            )
            raise RuntimeError(
                f"{area}-{product} prediction failed for {len(failures)} pixels: {preview}"
            )

display(pd.DataFrame(prediction_summaries))